# Fold Leak Guard

**Purpose**: run the difference-hash near-duplicate scan and rebuild leak-guard groups, then assert that no group key, raw Video ID, or near-duplicate pair straddles a fold boundary.

**Input**: `JPEGImages/`, `NoTools/`, and `ImageSets/default.txt` (local first, GCS fallback), the annotator sheet, and the baseline `folds.json`.

**Output**: dhash hit count, identifier-to-group collapse, leak-check assertions, per-fold group and frame counts.

## 1. Install dependencies
Run this cell to install the packages required for this notebook.

In [ ]:
%pip install -q tqdm pandas opencv-python-headless

import cv2, pandas as pd, numpy as _np
print(f"cv2 {cv2.__version__} | pandas {pd.__version__} | numpy {_np.__version__}")

## 2. Setup and configuration
Resolves `JPEGImages/`, `NoTools/`, `ImageSets/default.txt`, and `folds.json` (local first, GCS fallback), then sets the
annotator sheet. Override with environment variables if the files move.

In [ ]:
import os
import glob
import json
import re
import shutil
import subprocess
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm

# ── Paths ────────
# Override with environment variables if the files move.
GCS_BUCKET = os.getenv("LAP_GCS_BUCKET", "laparoscopic-seg-REPLACE_K_NUMBER")
GCS_PREFIX = f"gs://{GCS_BUCKET}/trainingFrames"
CACHE_DIR = Path(os.getenv("LAP_CACHE_DIR", Path.home() / ".cache" / "laparoscopic_seg"))

DEFAULT_DATA_ROOT = Path.home() / "Desktop" / "Individual Project" / "combined"
IMAGES_DIR = Path(os.getenv("LAP_IMAGES_DIR", DEFAULT_DATA_ROOT / "JPEGImages"))
NOTOOLS_DIR = Path(os.getenv("LAP_NOTOOLS_DIR", DEFAULT_DATA_ROOT / "NoTools"))
IMAGESETS_DIR = Path(os.getenv("LAP_IMAGESETS_DIR", DEFAULT_DATA_ROOT / "ImageSets" / "Segmentation"))
SHEET_ID = os.getenv("LAP_SHEET_ID", "REPLACE_SHEET_ID")
FOLDS_JSON_SRC = os.getenv(
    "LAP_FOLDS_JSON",
    "gs://training-checkpoints-REPLACE_K_NUMBER/experiments/"
    "baseline-seed2026_cv_Unet_resnet34_bce_lr1e-04_p20/20260812-205137/folds.json",
)
PHASH_HAMMING = 4  # differing bits (of 64) that count as a near-duplicate


def _count_jpgs(root: Path) -> int:
    return len(glob.glob(str(root / "**" / "*.jpg"), recursive=True))


def _gsutil_available() -> bool:
    return shutil.which("gsutil") is not None


def _gsutil_cp(src: str, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "cp", src, str(dst)], check=True)


def _gsutil_rsync(src: str, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "-m", "rsync", "-r", src, str(dst)], check=True)


def resolve_dir(local: Path, gcs_sub: str, required: bool = True) -> Path:
    if local.is_dir() and _count_jpgs(local) > 0:
        return local
    if not _gsutil_available():
        if not required:
            print(f"{gcs_sub} not found locally and gsutil is unavailable; continuing empty")
            return local
        raise FileNotFoundError(
            f"{gcs_sub} not found (or empty): {local}\n"
            "Set the matching LAP_*_DIR, or install gsutil to pull from GCS."
        )
    cached = CACHE_DIR / gcs_sub
    if _count_jpgs(cached) == 0:
        print(f"Downloading {gcs_sub} from {GCS_PREFIX}/{gcs_sub} ...")
        _gsutil_rsync(f"{GCS_PREFIX}/{gcs_sub}", cached)
    else:
        print(f"Using cached {gcs_sub} ({_count_jpgs(cached)} jpgs): {cached}")
    return cached


def resolve_folds_json() -> Path:
    if not FOLDS_JSON_SRC.startswith("gs://"):
        p = Path(FOLDS_JSON_SRC)
        if p.is_file():
            return p
        raise FileNotFoundError(f"LAP_FOLDS_JSON set but not found: {p}")
    cached = CACHE_DIR / "folds.json"
    if not cached.is_file():
        if not _gsutil_available():
            raise FileNotFoundError(
                "folds.json not cached and gsutil is unavailable. "
                "Set LAP_FOLDS_JSON to a local path, or install the Google Cloud SDK."
            )
        print(f"Downloading folds.json from {FOLDS_JSON_SRC} ...")
        _gsutil_cp(FOLDS_JSON_SRC, cached)
    else:
        print(f"Using cached folds.json: {cached}")
    return cached


def resolve_default_txt() -> Path:
    local = IMAGESETS_DIR / "default.txt"
    if local.is_file():
        return IMAGESETS_DIR
    if not _gsutil_available():
        raise FileNotFoundError(f"default.txt not found at {local}")
    cached = CACHE_DIR / "ImageSets" / "Segmentation"
    dest = cached / "default.txt"
    if not dest.is_file():
        print(f"Downloading default.txt from {GCS_PREFIX}/ImageSets/Segmentation ...")
        _gsutil_cp(f"{GCS_PREFIX}/ImageSets/Segmentation/default.txt", dest)
    return cached


IMAGES_DIR = resolve_dir(IMAGES_DIR, "JPEGImages")
NOTOOLS_DIR = resolve_dir(NOTOOLS_DIR, "NoTools", required=False)
IMAGESETS_DIR = resolve_default_txt()
FOLDS_JSON = resolve_folds_json()

n_img = _count_jpgs(IMAGES_DIR)
n_nt = _count_jpgs(NOTOOLS_DIR)
print("Images dir:", IMAGES_DIR)
print("NoTools dir:", NOTOOLS_DIR, f"({n_nt} jpg)")
print("ImageSets:", IMAGESETS_DIR)
print("Folds json:", FOLDS_JSON)
print(f"Sheet id: {SHEET_ID}")
print(f"JPEGImages: {n_img}")
assert n_img > 0, f"No .jpg images under {IMAGES_DIR}"
assert (IMAGESETS_DIR / "default.txt").is_file(), f"default.txt not found: {IMAGESETS_DIR}"
assert FOLDS_JSON.is_file(), f"folds.json not found: {FOLDS_JSON}"

## 3. Frame list and Video IDs
Read `default.txt` plus `NoTools/`, join the annotator sheet on `videoID/stem`, and take `Video ID` (parent folder if the sheet cell is empty). Retired statuses are kept, unlike training 6b.

In [ ]:
def frame_key(path):
    parts = str(path).split("/")
    return f"{parts[-2]}/{Path(parts[-1]).stem}"


img_paths = []
stems = [l.strip() for l in (IMAGESETS_DIR / "default.txt").read_text().splitlines() if l.strip()]
for stem in stems:
    p = IMAGES_DIR / (stem + ".jpg")
    if p.is_file():
        img_paths.append(str(p))
img_paths += glob.glob(str(NOTOOLS_DIR / "**" / "*.jpg"), recursive=True)
# Keep first path per videoID/stem so a NoTools copy of an annotated frame is dropped.
by_key = {}
for p in img_paths:
    by_key.setdefault(frame_key(p), p)
img_paths = list(by_key.values())
keys = [frame_key(p) for p in img_paths]

meta_df = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv")
meta_df.columns = meta_df.columns.str.strip()
meta_df["key"] = meta_df["File Name"].map(frame_key)
lookup = meta_df.drop_duplicates("key").set_index("key")


def sheet_video(k, img):
    if k in lookup.index and "Video ID" in lookup.columns:
        v = lookup.at[k, "Video ID"]
        if not (isinstance(v, float) and np.isnan(v)) and str(v).strip():
            return str(v)
    return Path(img).parent.name


video_ids = np.array([sheet_video(k, img) for k, img in zip(keys, img_paths)])
print(f"frames         : {len(img_paths)}")
print(f"sheet match    : {sum(k in lookup.index for k in keys)}/{len(keys)}")
print(f"video ids      : {len(set(video_ids))}")

## 4. Difference-hash near-duplicate scan
Fingerprint each frame as a 64-bit code from an 8x9 greyscale thumbnail. Pairs within `PHASH_HAMMING` bits are near-duplicates. Copied from `00_model_training` section 6d.

In [ ]:
def _dhash(path):
    im = cv2.imread(path, cv2.IMREAD_REDUCED_GRAYSCALE_8)
    if im is None:
        im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    thumb = cv2.resize(im, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    brighter_than_left = thumb[:, 1:] > thumb[:, :-1]
    return np.packbits(brighter_than_left), float(thumb.std())


def _pairs_within_hamming(packed_hashes, frame_idxs, max_dist):
    bit_vecs = np.unpackbits(np.stack(packed_hashes)[frame_idxs], axis=1).astype(np.float32)
    n_ones = bit_vecs.sum(1)
    found = []
    for start in range(0, len(bit_vecs), 512):
        block = bit_vecs[start:start + 512]
        dist = n_ones[start:start + 512, None] + n_ones[None, :] - 2.0 * (block @ bit_vecs.T)
        rows, cols = np.where(np.rint(dist) <= max_dist)
        once = cols > (start + rows)
        found.append(np.stack([
            frame_idxs[start + rows[once]],
            frame_idxs[cols[once]],
        ], 1))
    return np.concatenate(found) if found else np.zeros((0, 2), int)


hashes, stds = zip(*(_dhash(p) for p in tqdm(img_paths, desc="Hashing frames", unit="img")))
stds = np.array(stds)
usable = np.where(stds >= 2.0)[0]
if len(usable) < len(img_paths):
    print(f"phash skip     : {len(img_paths) - len(usable)} near-featureless frames excluded")
_dupe_pairs = _pairs_within_hamming(hashes, usable, PHASH_HAMMING)
print(f"near-dupes     : {len(_dupe_pairs)} pairs (Hamming <= {PHASH_HAMMING})")
for i, j in _dupe_pairs[:8]:
    print(f"  e.g.         : {video_ids[i]}/{Path(img_paths[i]).stem}  ==  "
          f"{video_ids[j]}/{Path(img_paths[j]).stem}")
if len(_dupe_pairs) > 8:
    print(f"  ...          : {len(_dupe_pairs) - 8} more")

## 5. Leak-guard groups and fold assertions
Union-find over Video ID, export directory (skipping shared containers above 5%), and cross-group dhash pairs. Then load `folds.json` and assert that no group key, raw Video ID, or near-duplicate pair straddles a fold boundary.

In [ ]:
def _normalise(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).strip().lower())


_parent = {}

def _find(x):
    _parent.setdefault(x, x)
    r = x
    while _parent[r] != r:
        r = _parent[r]
    while _parent[x] != r:
        _parent[x], x = r, _parent[x]
    return r

def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        _parent[ra] = rb


dir_key = np.array([_normalise(Path(p).parent.name) for p in img_paths])
vid_key = np.array([_normalise(v) for v in video_ids])

dir_n = Counter(dir_key)
dir_ids = defaultdict(set)
for d, v in zip(dir_key, vid_key):
    dir_ids[d].add(v)
container = {d for d in dir_n
             if dir_n[d] > 0.05 * len(img_paths)
             or len(dir_ids[d]) > 0.05 * len(set(vid_key))}
if container:
    print(f"containers     : {sorted(container)}")

for d, v in zip(dir_key, vid_key):
    _find(("vid", v))
    if d not in container:
        _union(("vid", v), ("dir", d))

n_raw = len(set(video_ids))
cross = np.array([
    p for p in _dupe_pairs
    if _find(("vid", vid_key[p[0]])) != _find(("vid", vid_key[p[1]]))
]) if len(_dupe_pairs) else np.zeros((0, 2), int)
for i, j in cross:
    _union(("vid", vid_key[i]), ("vid", vid_key[j]))

roots = [_find(("vid", v)) for v in vid_key]
members = defaultdict(list)
for r, v in zip(roots, video_ids):
    members[r].append(v)
label = {r: Counter(vs).most_common(1)[0][0] for r, vs in members.items()}
video_group = np.array([label[r] for r in roots])

n_groups = len(set(video_group))
print(f"near-dupes     : {len(_dupe_pairs)} pairs, {len(cross)} cross-group")
print(f"leak groups    : {n_raw} Video IDs -> {n_groups} groups "
      f"({n_raw - n_groups} identifiers merged)")
assert max(Counter(video_group).values()) / len(img_paths) < 0.30
assert n_groups >= 15

saved = json.loads(Path(FOLDS_JSON).read_text())
key_to_idx = {k: i for i, k in enumerate(keys)}
missing = [k for i in range(5) for k in saved[str(i)] if k not in key_to_idx]
assert not missing, f"{len(missing)} fold keys not in the frame list, e.g. {missing[:3]}"
folds = [np.array([key_to_idx[k] for k in saved[str(i)]]) for i in range(5)]
print(f"folds loaded   : {FOLDS_JSON}")

for i in range(5):
    rest = np.concatenate([folds[j] for j in range(5) if j != i])
    assert not (set(video_group[folds[i]]) & set(video_group[rest])), f"group leak in fold {i}"
    assert not (set(video_ids[folds[i]]) & set(video_ids[rest])), f"Video ID leak in fold {i}"
if len(_dupe_pairs):
    fold_of = np.full(len(img_paths), -1, dtype=int)
    for i in range(5):
        fold_of[folds[i]] = i
    both_in = (fold_of[_dupe_pairs[:, 0]] >= 0) & (fold_of[_dupe_pairs[:, 1]] >= 0)
    straddle = int((fold_of[_dupe_pairs[both_in, 0]] != fold_of[_dupe_pairs[both_in, 1]]).sum())
    assert straddle == 0, f"{straddle} near-duplicate frame pairs straddle folds"
print("leak check     : no video leaks across folds "
      "(group key, Video ID, near-duplicate pixels)")

rows = []
for i in range(5):
    idx = folds[i]
    rows.append({
        "fold": i,
        "frames": len(idx),
        "groups": len(set(video_group[idx])),
        "video_ids": len(set(video_ids[idx])),
    })
print(pd.DataFrame(rows).to_string(index=False))